# Bose2 — Free GPU video experiment (manual, optional)

This notebook is an **experiment only**, not an automated production service. Open it in Google Colab, choose a GPU runtime only if the free option is offered, and run cells manually. Do not purchase credits, upgrade, or enable billing. Free GPU access is not guaranteed.

It tries one short CogVideoX-2B text-to-video clip. Generation may fail or run out of memory on a free GPU. That is an acceptable test result; the Bose2 factory remains unchanged.


In [ ]:
import torch, platform, os, json
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is available in this runtime. Stop here; do not upgrade or enable paid compute.')
gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print('GPU:', gpu)
print(f'VRAM: {vram_gib:.1f} GiB')
print('This notebook does not configure billing or call any paid API.')


## 1. Install experiment dependencies

This installs Python packages into the temporary notebook runtime. It does not install anything on your phone or computer.

In [ ]:
%pip -q install --upgrade diffusers transformers accelerate imageio imageio-ffmpeg

## 2. Generate one short clip

The prompt asks for a crochet scene, but generative video can make inaccurate fingers, hooks, yarn, and stitches. Treat this as a visual test, not as proof of correct crochet instruction.

In [ ]:
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
from pathlib import Path
import torch, time

model_id = 'zai-org/CogVideoX-2b'
prompt = (
    'A realistic close-up craft tutorial scene on a clean table: an adult woman '
    'demonstrator calmly holds a crochet hook and soft cotton yarn, making a slow '
    'simple crochet motion, warm natural studio light, stable camera, clear hands, '
    'educational video, no text, no logos.'
)

pipe = CogVideoXPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

started = time.time()
with torch.inference_mode():
    result = pipe(
        prompt=prompt,
        num_videos_per_prompt=1,
        num_inference_steps=20,
        num_frames=49,
        guidance_scale=6,
        generator=torch.Generator(device='cpu').manual_seed(42),
    )

frames = result.frames[0]
out = Path('/content/bose2_cogvideox_test.mp4')
export_to_video(frames, str(out), fps=8)
print('Saved:', out, '| bytes:', out.stat().st_size, '| seconds:', round(time.time()-started, 1))
assert out.is_file() and out.stat().st_size > 100_000, 'Video artifact missing or unexpectedly small.'


## 3. Review before considering integration

Play the MP4 and check: (1) is it genuinely moving, (2) do the hands/hook/yarn stay coherent, (3) is the crochet action understandable, and (4) did the run finish without an out-of-memory error? If any answer is no, do not integrate this model into production yet.

In [ ]:
from IPython.display import Video, display
display(Video('/content/bose2_cogvideox_test.mp4', embed=True))

## 4. Download the test artifact

Use the download control on the video preview or run the cell below. The file is temporary and should be downloaded before the runtime ends.

In [ ]:
from google.colab import files
files.download('/content/bose2_cogvideox_test.mp4')